# Shortcut check · RoBERTa on normalised text (3 seeds, no augmentation)

Settings: **Accelerator GPU T4** (one GPU is enough), **Internet On**. Input: the `raid-processed`
dataset, new version containing `normalize_text.py`.
Run with **Save Version → Save & Run All (Commit)**. About 1.5 hours.

In [1]:
# Copy scripts and data from the raid-processed dataset into a working project folder
import glob, os, shutil
# (the folder that has both the data and the scripts, i.e. the dataset itself -
#  not a copy inside an earlier notebook's output)
roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train_clean.parquet", recursive=True)]
roots = [r for r in roots if glob.glob(f"{r}/**/normalize_text.py", recursive=True)]
assert roots, "raid-processed dataset (with normalize_text.py) not found - check the Input panel"
data_root = roots[0]
os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/data/processed", exist_ok=True)
for f in glob.glob(f"{data_root}/**/*.py", recursive=True):
    shutil.copy(f, "/kaggle/working/src/")
for f in glob.glob(f"{data_root}/**/*.parquet", recursive=True):
    shutil.copy(f, "/kaggle/working/data/processed/")
os.chdir("/kaggle/working")
print("src:", sorted(os.listdir("src")))
print("data:", sorted(os.listdir("data/processed")))
for need in ["normalize_text.py", "metrics.py", "train_roberta.py", "train_classical.py"]:
    assert need in os.listdir("src"), f"{need} missing - upload it to the dataset (New Version)"
!pip uninstall -y -q torchao   # Kaggle's old torchao breaks recent peft; not needed here
!pip install -q peft
!nvidia-smi --query-gpu=name,memory.total --format=csv

src: ['inspect_raid.py', 'make_splits.py', 'metrics.py', 'normalize_text.py', 'train_classical.py', 'train_llm.py', 'train_roberta.py', 'zero_shot_binoculars.py']
data: ['test_heldout_attacks.parquet', 'test_reppen.parquet', 'test_seen.parquet', 'test_seen_attacks.parquet', 'test_unseen.parquet', 'train_attacked.parquet', 'train_clean.parquet', 'val.parquet']
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 1. Remove formatting clues from every text (about 1 minute)

In [2]:
!python src/normalize_text.py --src data/processed --dst data/processed_norm

test_heldout_attacks.parquet    20,060 texts | changed  97.0% | had line breaks  77.4%
test_reppen.parquet              4,012 texts | changed  85.3% | had line breaks  69.8%
test_seen.parquet                4,012 texts | changed  85.6% | had line breaks  72.0%
test_seen_attacks.parquet       24,072 texts | changed  68.9% | had line breaks  59.7%
test_unseen.parquet              4,012 texts | changed  85.3% | had line breaks  73.1%
train_attacked.parquet          18,718 texts | changed  68.5% | had line breaks  59.2%
train_clean.parquet             18,718 texts | changed  85.4% | had line breaks  71.9%
val.parquet                      4,012 texts | changed  86.2% | had line breaks  72.5%

Most common non-ASCII characters left in held-out-attack texts
(homoglyphs that are still here are not in the mapping):
   ('’', 'U+2019', 'RIGHT SINGLE QUOTATION MARK', 1474)
   ('–', 'U+2013', 'EN DASH', 354)
   ('£', 'U+00A3', 'POUND SIGN', 321)
   ('é', 'U+00E9', 'LATIN SMALL LETTER E WITH ACUTE', 

## 2. Quick check (a few minutes; numbers are meaningless)

In [3]:
!python src/train_roberta.py --data data/processed_norm --limit 300 --seeds 42 --aug 0 --epochs 1 --out results/smoke_norm

Device: cuda
Tokenizing...
config.json: 100%|█████████████████████████████| 481/481 [00:00<00:00, 1.97MB/s]
tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 148kB/s]
vocab.json: 100%|████████████████████████████| 899k/899k [00:00<00:00, 29.2MB/s]
merges.txt: 100%|████████████████████████████| 456k/456k [00:00<00:00, 4.53MB/s]
tokenizer.json: 100%|██████████████████████| 1.36M/1.36M [00:00<00:00, 13.0MB/s]
  20,530 texts tokenized
Run aug=0.0 lr=2e-05 seed=42

model.safetensors: downloading bytes: ██████▎               |  142MB, 10.6MB/s  
model.safetensors: downloading bytes: ████████████▊         |  290MB, 23.6MB/s  
model.safetensors: downloading bytes: ██████████████▊       |  335MB, 30.0MB/s  
model.safetensors: downloading bytes: ██████████████████████|  335MB, 31.1MB/s  
model.safetensors: reconstructing file: 100%|██████|  499MB /  499MB, 47.1MB/s  
Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 2674.13it/s]
[transformers] RobertaForSe

## 3. RoBERTa on normalised text, same settings as the original runs (lr 2e-5)

In [4]:
!python src/train_roberta.py --data data/processed_norm --aug 0 --lrs 2e-5 --out results/normalized

Device: cuda
Tokenizing...
  93,604 texts tokenized
Run aug=0.0 lr=2e-05 seed=42
Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 2703.53it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
      epoch 1 batch 50/2340  loss 0.70